# 17 · Probe layer validity + probe–binary divergence (standalone Exp 5/6)

Runs **only** experiments 5 and 6 from notebook 16, against whatever probe / battery-rows /
shift files are currently on Drive — so it works on the *old* (v1) organisms today, before the
retrain metas, and on the -2 organisms afterwards (where it shares 16's activation cache and is
nearly instant if 16 already ran).

**Exp 5 — layer-specific probe construct validity:** the 06b probe exists at every layer 16–34;
compare cos(probe, dark-specific) and probe→willingness / probe→binary prediction across depth.
**Exp 6 — probe–binary divergence:** sort dark-triad items by z(probe) − z(binary), read the
tails, aggregate by subscale.

**To run on the v1 organisms** (while their files are still on Drive — meta_1's cleanup deletes
them): set `RUN_TAG = "_v1"` below and point `dark` at `Koalacrown/dark-qwen3-8b-rl-merged` in
the config cell. Only the dark organism's activations are extracted (~10 min on L4).

## 1. Setup

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
%pip install -q -U "numpy>=2.1" "scipy>=1.13" scikit-learn transformers accelerate sentencepiece
import sys, importlib
for _m in ("numpy","scipy","sklearn","transformers"):
    importlib.import_module(_m); print(_m, "->", getattr(sys.modules[_m], "__version__", "ok"))

In [ ]:
import pathlib
DRIVE = mount_drive()
use_probe_repo()
RUN_TAG = ""      # "" = current organisms (shares item_acts_v1/components_v1 with notebook 16).
                  # "_v1" = old organisms -> separate item_acts_v1_v1 / components_v1_v1 dirs,
                  # so the caches can't collide with the -2 retrain's.
DIRS  = (DRIVE / "directions_v1")             if DRIVE else pathlib.Path("directions_v1")
ACTS  = (DRIVE / f"item_acts_v1{RUN_TAG}")    if DRIVE else pathlib.Path(f"item_acts_v1{RUN_TAG}")
OUT   = (DRIVE / f"components_v1{RUN_TAG}")   if DRIVE else pathlib.Path(f"components_v1{RUN_TAG}")
for p in (ACTS, OUT): p.mkdir(parents=True, exist_ok=True)

BATTERY_DIR = None
for ver in ("battery_v5", "battery_v4"):
    cand = (DRIVE / ver) if DRIVE else pathlib.Path(ver)
    if (cand / "rows_dark.csv").exists():
        BATTERY_DIR = cand
        if ver == "battery_v4" and not RUN_TAG:
            print("!! battery_v4 rows = old-organism scores; fine for RUN_TAG='_v1', not for -2.")
        break
assert BATTERY_DIR is not None, "no battery rows found — run notebook 09 first"
assert (DIRS / "control_vectors_shift_dark.pkl").exists(), "shift vectors missing — run 06c"
assert (DIRS / "probe_dark_all.npz").exists(), "probe missing — run 06b"
print("directions <-", DIRS, "| battery <-", BATTERY_DIR, "| acts ->", ACTS, "| out ->", OUT)

## 2. Config
Only `dark` needs activations here (Exp 5/6 never touch base or depression acts); the shift
pickles supply both organisms' directions. For the v1 run swap the `hf` id (see header).

In [ ]:
ORGANISMS = [
    {"name": "dark", "hf": "Koalacrown/dark-2-qwen3-8b"},   # v1: Koalacrown/dark-qwen3-8b-rl-merged
]
ACT_LAYERS = list(range(16, 25)) + list(range(30, 35))
PROBE_L    = 18            # battery probe layer (09)
SELECTOR   = "task_mean"
BATCH      = 16
MAXTOK     = 512
SKIP_EXISTING = True
BANDS      = {"mid (16-24)": range(16, 25), "late (30-34)": range(30, 35)}
print(f"{len(ORGANISMS)} organisms | layers {ACT_LAYERS} | selector {SELECTOR}")

## 3. Items + battery scores
Battery items from `data/source_items/*.jsonl` (dark-triad instruments carry `trait`,
internalizing ones carry `mechanism`), generalization requests from `data/probe_generalization/`.
Scores join on `id` from the 09 rows CSVs — `binary_endorse` is already sign-corrected there.

In [ ]:
import json, glob, csv, collections

def load_jsonl(p):
    return [json.loads(l) for l in open(p) if l.strip()]

ITEMS = {}                       # id -> item dict (+ "side": "trait"|"mechanism", "instrument")
for f in sorted(glob.glob("/content/dt_rl/data/source_items/*.jsonl")):
    inst = pathlib.Path(f).stem
    for it in load_jsonl(f):
        it["instrument_file"] = inst
        it["side"] = "trait" if "trait" in it else "mechanism"
        ITEMS[it["id"]] = it
GEN = {}                         # id -> {category, text}
for f in sorted(glob.glob("/content/dt_rl/data/probe_generalization/*.jsonl")):
    for it in load_jsonl(f):
        GEN[it["id"]] = it

ROWS = {}                        # organism -> {id: row}
for spec in ORGANISMS:
    fp = BATTERY_DIR / f"rows_{spec['name']}.csv"
    if fp.exists():
        ROWS[spec["name"]] = {r["id"]: r for r in csv.DictReader(open(fp))}
    else:
        print(f"!! rows_{spec['name']}.csv missing — Exp 1-3 will skip this organism")

# ordered id lists (battery items must exist in source files; gen ids from probe_generalization)
BAT_IDS = [i for i in ROWS.get("dark", ROWS.get("base", {})) if i in ITEMS]
GEN_IDS = [i for i in ROWS.get("dark", ROWS.get("base", {})) if i in GEN]
ALL_IDS = BAT_IDS + GEN_IDS
TEXTS   = {**{i: ITEMS[i]["text"] for i in BAT_IDS}, **{i: GEN[i]["text"] for i in GEN_IDS}}
print(f"{len(BAT_IDS)} battery items | {len(GEN_IDS)} gen items | "
      f"sides: {collections.Counter(ITEMS[i]['side'] for i in BAT_IDS)}")

## 4. Per-item activations (the missing artifact)
One model load per organism; `get_activations_batch` on the **bare item text** (same
administration as 09's probe readout: single user message, no scale framing), `task_mean`
pooling at all `ACT_LAYERS`. Saved to `item_acts_v1/acts_items_<name>.npz` (fp16, ~75 MB each).

In [ ]:
import numpy as np, torch, gc
from tqdm.auto import tqdm
from src.models.huggingface_model import HuggingFaceModel

def extract_org(spec):
    name = spec["name"]; fp = ACTS / f"acts_items_{name}.npz"
    if SKIP_EXISTING and fp.exists():
        print(f"[skip] {name} (cached)"); return
    print(f"[load] {name} <- {spec['hf']}")
    model = HuggingFaceModel(spec["hf"], dtype="bfloat16", device="cuda")
    model.tokenizer.padding_side = "left"
    X = {L: [] for L in ACT_LAYERS}
    ids = list(ALL_IDS)
    for i in tqdm(range(0, len(ids), BATCH), desc=name):
        chunk = ids[i:i+BATCH]
        msgs = []
        for iid in chunk:
            t = TEXTS[iid]
            tok_ids = model.tokenizer(t, add_special_tokens=False).input_ids
            if len(tok_ids) > MAXTOK:
                t = model.tokenizer.decode(tok_ids[:MAXTOK])
            msgs.append([{"role": "user", "content": t}])
        res = model.get_activations_batch(msgs, ACT_LAYERS, [SELECTOR])
        for L in ACT_LAYERS:
            X[L].append(np.asarray(res[SELECTOR][L], dtype=np.float16))
    np.savez_compressed(fp, ids=np.array(ids),
                        **{f"L{L}": np.concatenate(X[L]) for L in ACT_LAYERS})
    print(f"[done] {name} -> {fp.name}")
    del model; gc.collect(); torch.cuda.empty_cache()

for spec in ORGANISMS:
    extract_org(spec)

def load_acts(name):
    z = np.load(ACTS / f"acts_items_{name}.npz")
    ids = list(z["ids"])
    idx = {i: j for j, i in enumerate(ids)}
    return {L: z[f"L{L}"].astype(np.float32) for L in ACT_LAYERS}, idx

ACT, IDX = {}, {}
for spec in ORGANISMS:
    ACT[spec["name"]], IDX[spec["name"]] = load_acts(spec["name"])
print("activations in memory:", list(ACT))

## 5. Component vectors
Same math as 15 cell 8, both directions:
`shared_L = (dark_L · û_dep_L) û_dep_L`, `residual_L = dark_L − shared_L` (dark-specific), and
symmetrically `dep_residual_L = dep_L − (dep_L · û_dark_L) û_dark_L` (depression-specific).

In [ ]:
import pickle

def load_shift(name):
    return pickle.load(open(DIRS / f"control_vectors_shift_{name}.pkl", "rb"))["vectors"]["induced_shift"]

dark_s, dep_s = load_shift("dark"), load_shift("clinical-depression")
SHIFT_LAYERS = sorted(set(map(int, dark_s)) & set(map(int, dep_s)))
COMP = {}   # {L: {"shared", "residual", "dep_residual", "dark", "depression"}}
for L in SHIFT_LAYERS:
    a = np.asarray(dark_s[L], np.float32); b = np.asarray(dep_s[L], np.float32)
    u_dep, u_dark = b / np.linalg.norm(b), a / np.linalg.norm(a)
    shared = float(a @ u_dep) * u_dep
    COMP[L] = {"shared": shared, "residual": a - shared,
               "dep_residual": b - float(b @ u_dark) * u_dark,
               "dark": a, "depression": b}
CL = [L for L in ACT_LAYERS if L in COMP]
print(f"shift layers {SHIFT_LAYERS[0]}..{SHIFT_LAYERS[-1]} | usable with acts: {CL}")

## 6. Exp 5 — layer-specific probe construct validity
06b saved the probe at **every** layer of the 0.45–0.95 band (per-layer `unit`/`w_raw`/`r` in
`probe_dark_all.npz`) — L18 is just the battery's pick. Per layer: cos(probe, dark-specific /
shared), probe-score → willingness on the dark requests, probe-score → binary endorsement on
positively-keyed dark-triad items, each also as a semi-partial controlling the shared projection.

Reading: **MID keeps `cos_residual` / `sr_will` while LATE loses them** → the probe's construct
validity depends on reading *before* J-space absorbs the dark-specific component into the shared
output pathway. **Flat across depth** → probe validity is not tied to the J-space geometry.
(Positively-keyed items only: `probe_raw` scores the raw text, `binary_endorse` is sign-corrected,
so reverse-keyed items would anti-align the two readouts by construction.)

In [ ]:
from scipy import stats as st
pz = np.load(DIRS / "probe_dark_all.npz")
p_layers = list(map(int, pz["layers"]))

def zsc(x):
    x = np.asarray(x, float); return (x - x.mean()) / (x.std() + 1e-12)

def proj_scores(org, ids, L, vec):
    u = vec / np.linalg.norm(vec)
    return ACT[org][L][[IDX[org][i] for i in ids]] @ u

def cosv(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

def probe_scores(org, ids, L):
    j = p_layers.index(L)
    x = ACT[org][L][[IDX[org][i] for i in ids]]
    return ((x - pz["mean"][j]) / pz["scale"][j]) @ pz["w_raw"][j] + pz["b_raw"][j]

dt_pos = [i for i in BAT_IDS if ITEMS[i]["side"] == "trait"
          and str(ROWS["dark"][i].get("is_filler", "False")) != "True"
          and float(ROWS["dark"][i].get("sign", 1) or 1) > 0
          and ROWS["dark"][i]["binary_endorse"] not in ("", None)]
y_bin = np.array([float(ROWS["dark"][i]["binary_endorse"]) for i in dt_pos])
will = {i: float(ROWS["dark"][i]["willingness"]) for i in GEN_IDS
        if ROWS["dark"][i]["willingness"] not in ("", None)}
dark_req = [i for i in will if GEN[i]["category"] == "dark"]
y_will = np.array([will[i] for i in dark_req])
print(f"{len(dt_pos)} pos-keyed dark-triad items | {len(dark_req)} dark requests\n")

P_USE = [L for L in CL if L in p_layers]
EXP5 = []
print("layer  heldout_r  cos_res   cos_sh   r_will  sr_will|sh    r_bin  sr_bin|sh")
for L in P_USE:
    j = p_layers.index(L)
    w = pz["unit"][j].astype(np.float32)
    row = {"layer": L, "heldout_r_mu": float(pz["r"][j]),
           "cos_residual": cosv(w, COMP[L]["residual"]),
           "cos_shared":   cosv(w, COMP[L]["shared"])}
    for tag, ids, y in (("will", dark_req, y_will), ("bin", dt_pos, y_bin)):
        ps = zsc(probe_scores("dark", ids, L))
        sh = zsc(proj_scores("dark", ids, L, COMP[L]["shared"]))
        row[f"r_{tag}"]  = st.pearsonr(ps, y)[0]
        resid = ps - np.polyval(np.polyfit(sh, ps, 1), sh)
        row[f"sr_{tag}"] = st.pearsonr(zsc(resid), y)[0]
    EXP5.append(row)
    print(f"  L{L:2d} {row['heldout_r_mu']:+9.3f} {row['cos_residual']:+8.3f} {row['cos_shared']:+8.3f} "
          f"{row['r_will']:+8.3f} {row['sr_will']:+11.3f} {row['r_bin']:+8.3f} {row['sr_bin']:+10.3f}")

for bname, rng_ in BANDS.items():
    rs = [r for r in EXP5 if r["layer"] in rng_]
    if rs:
        print(f"  {bname} mean:  " + "  ".join(
            f"{k}={np.mean([r[k] for r in rs]):+.3f}"
            for k in ("cos_residual", "cos_shared", "r_will", "sr_will", "r_bin")))

with open(OUT / "exp5_probe_layers.json", "w") as f:
    json.dump(EXP5, f, indent=2)
print("\nsaved ->", OUT / "exp5_probe_layers.json")

## 7. Exp 6 — item-level probe–binary divergence (sorting, no compute)
Rank the positively-keyed dark-triad items by `z(probe) − z(binary_endorse)` on the dark
organism and read both tails, then aggregate by subscale. Uses 09's own `probe_raw` column when
present (the battery's L18 administration); otherwise scores items with the 06b probe on our
activations. If the "probe high / binary low" tail is the ego-syntonic subscales (admiration,
boldness, meanness), the probe is seeing wanting exactly where verbal self-report goes blind —
converging with Exp 1 and Exp 5.

In [ ]:
have_col = any(ROWS["dark"][i].get("probe_raw") not in ("", None) for i in dt_pos)
if have_col:
    p_src = {i: float(ROWS["dark"][i]["probe_raw"]) for i in dt_pos
             if ROWS["dark"][i].get("probe_raw") not in ("", None)}
    print(f"using 09's probe_raw column ({len(p_src)}/{len(dt_pos)} items)")
else:
    p_src = dict(zip(dt_pos, map(float, probe_scores("dark", dt_pos, PROBE_L))))
    print("probe_raw column empty — scoring items with the 06b probe on our activations")
d6_ids = [i for i in dt_pos if i in p_src]

zp = zsc([p_src[i] for i in d6_ids])
zb = zsc([float(ROWS["dark"][i]["binary_endorse"]) for i in d6_ids])
div = zp - zb
r_pb = st.pearsonr(zp, zb)[0]
print(f"r(probe, binary) over {len(d6_ids)} pos-keyed dark-triad items: {r_pb:+.3f}\n")

def item_label(i):
    it = ITEMS[i]
    sub = it.get("subscale") or it.get("trait") or ROWS["dark"][i].get("cat_or_group", "")
    return f"{it['instrument_file']:9s} {str(sub)[:14]:14s} {it['text'][:64]}"

order = np.argsort(div)
print("probe HIGH / binary LOW (probe sees wanting that the endorsement denies):")
for k in order[::-1][:12]:
    print(f"  {div[k]:+5.2f}  {item_label(d6_ids[k])}")
print("\nprobe LOW / binary HIGH (endorsed but not wanted):")
for k in order[:12]:
    print(f"  {div[k]:+5.2f}  {item_label(d6_ids[k])}")

groups = collections.defaultdict(list)
for k, i in enumerate(d6_ids):
    it = ITEMS[i]
    groups[(it["instrument_file"], str(it.get("subscale") or it.get("trait") or ""))].append(float(div[k]))
print("\nmean divergence by subscale (n>=4) — systematic if ordered, item noise if not:")
EXP6_GROUPS = []
for (inst, sub), vals in sorted(groups.items(), key=lambda kv: -np.mean(kv[1])):
    if len(vals) < 4: continue
    EXP6_GROUPS.append({"instrument": inst, "subscale": sub, "n": len(vals),
                        "mean_div": float(np.mean(vals))})
    print(f"  {np.mean(vals):+5.2f}  (n={len(vals):2d})  {inst}/{sub}")

with open(OUT / "exp6_probe_binary_divergence.json", "w") as f:
    json.dump({"probe_source": "09_probe_raw" if have_col else "06b_probe_on_acts",
               "r_probe_binary": float(r_pb), "groups": EXP6_GROUPS,
               "items": [{"id": i, "div": float(div[k]), "probe_z": float(zp[k]),
                          "binary_z": float(zb[k])} for k, i in enumerate(d6_ids)]}, f, indent=2)
print("\nsaved ->", OUT / "exp6_probe_binary_divergence.json")

---
# Done
`exp5_probe_layers.json` + `exp6_probe_binary_divergence.json` in the (tagged) `components_v1`
dir. Ran on v1? Compare against the -2 numbers once meta_3 has produced them.